In [ ]:
import pandas as pd

path_to_data = "./data/cleaner/centerlines_data.json"

cldata = pd.read_json(path_to_data)
#cldata.head()

,ROADNAME,CORE_CLASS,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,SIFIDHI,HICROSSSIF,GEOMETRY
1,SERENITY CT,LOCAL,8665,9887,1550,1557,8594,9811,"[[-85.6809503218, 38.1588670887], [-85.6810007..."
2,S 28TH ST,LOCAL,5926,6570,2854,2934,2470,2499,"[[-85.8012012237, 38.230563793], [-85.80136926..."
3,BEECH ST,LOCAL,473,0458,6487,7212,10596,D596,"[[-85.8050149882, 38.2289330216], [-85.8048524..."
4,GARDEN DR,PRIMARY COLLECTOR,2442,2470,13394,5391,4702,5128,"[[-85.6802054534, 38.2480671366], [-85.6798630..."
5,PARKWAY DR,LOCAL,4573,4974,4162,4476,8594,9811,"[[-85.7420397977, 38.2118147709], [-85.7416475..."
...,...,...,...,...,...,...,...,...,...
180232,BROOKE ELIZABETH WAY,LOCAL,15329,F540,8594,9811,15331,F541,"[[-85.7085947307, 38.1432988961], [-85.7081273..."
180233,AIKEN RIDGE DR,LOCAL,15611,F760,15610,F759,15642,F791,"[[-85.4509340541, 38.2704398554], [-85.4510619..."
180234,AIKEN RIDGE DR,LOCAL,15611,F760,15642,F791,15610,F759,"[[-85.4517479832, 38.2709209675], [-85.4518987..."
180235,AIKEN RIDGE CIR,LOCAL,15610,F759,15611,F760,15611,F760,"[[-85.4527137888, 38.2712854193], [-85.4523040..."


In [262]:
# GEOMETRY is recorded as lists of lists
# this causesproblems later b/c one algo uses a hash map and lists are unhashable
#TODO Perhaps fix this in the JSON import or in the cleaned file? Look at JSON data types

def fix_geo(points:[[float, float]]) -> ((float, float),):
    return tuple((long, lat) for long, lat in points)

cldata.GEOMETRY = cldata.GEOMETRY.apply(fix_geo)
cldata.head()

,ROADNAME,CORE_CLASS,SIFID,SIFCODE,SIFIDLOW,LOCROSSSIF,SIFIDHI,HICROSSSIF,GEOMETRY
1,SERENITY CT,LOCAL,8665,9887,1550,1557,8594,9811,"((-85.6809503218, 38.1588670887), (-85.6810007..."
2,S 28TH ST,LOCAL,5926,6570,2854,2934,2470,2499,"((-85.8012012237, 38.230563793), (-85.80136926..."
3,BEECH ST,LOCAL,473,0458,6487,7212,10596,D596,"((-85.8050149882, 38.2289330216), (-85.8048524..."
4,GARDEN DR,PRIMARY COLLECTOR,2442,2470,13394,5391,4702,5128,"((-85.6802054534, 38.2480671366), (-85.6798630..."
5,PARKWAY DR,LOCAL,4573,4974,4162,4476,8594,9811,"((-85.7420397977, 38.2118147709), (-85.7416475..."


In [263]:
USE_SIFID_COLUMNS = ['ROADNAME', 'SIFID', 'SIFIDLOW', 'SIFIDHI', 'GEOMETRY']
USE_SIFCODE_COLUMNS = ['ROADNAME', 'SIFCODE', 'LOCROSSSIF', 'HICROSSSIF', 'GEOMETRY']


In [264]:
SIFID_lookup = pd.Series(cldata.groupby(by='SIFID').indices, name='Lookup by SIFID')
SIFID_lookup.index.name='SIFID'
SIFID_lookup

SIFCODE_lookup = pd.Series(cldata.groupby(by='SIFCODE').indices, name='Lookup by SIFCODE')
SIFCODE_lookup.index.name='SIFCODE'
SIFCODE_lookup

# data.groupby(by='SIFID')['SIFCODE'].apply(lambda xs:(len(set(xs)) <= 1)).all() # == True
# data.groupby(by='SIFCODE')['SIFID'].apply(lambda xs:(len(set(xs)) <= 1)).all() # == True
# unique mapping between SIFID and SIFCODE. Good news. 

SIFID_lookup



SIFID
1        [2170, 3212, 5879, 7816, 8127, 9619, 10394, 10...
2                                                  [34888]
3                                            [2393, 27523]
4                                                   [2557]
5                                                  [25529]
                               ...                        
15638                                       [35020, 35021]
15639                                              [35024]
15640                                              [35029]
15641                                              [35031]
15642                                              [35038]
Name: Lookup by SIFID, Length: 11347, dtype: object

In [265]:
counts = SIFID_lookup.apply(len).sort_values(ascending=True)
SIFID_lookup[counts > 1]

# if you don't care about long vs short multisegments, which I don't think we do.
multisegments = SIFID_lookup[counts > 1].index

# for a list of multi segment roads sorted by list length
SIFID_lookup[counts[counts > 1].index]

SIFID_lookup[counts>1]

#cldata[cldata.SIFID==2543]

SIFID
1        [2170, 3212, 5879, 7816, 8127, 9619, 10394, 10...
3                                            [2393, 27523]
8                             [16342, 24413, 29597, 30362]
9                                     [5193, 10855, 27942]
10               [4502, 19690, 20238, 21195, 27977, 30826]
                               ...                        
15624                                       [34966, 34970]
15626                         [34975, 34976, 34978, 34981]
15628                                       [34993, 35018]
15632                  [35002, 35003, 35006, 35007, 35011]
15638                                       [35020, 35021]
Name: Lookup by SIFID, Length: 5763, dtype: object

In [266]:
from operator import itemgetter

sifid_data = cldata[USE_SIFID_COLUMNS].copy()

first_last_points = sifid_data.GEOMETRY.transform({"GEOFIRST":itemgetter(0), "GEOLAST":itemgetter(-1)})

sifid_data = pd.concat((sifid_data, first_last_points), axis=1).drop("GEOMETRY", axis=1)

for name in ['next_id', 'prev_id', 'low_id', 'hi_id']:
    sifid_data[name] = pd.Series()

sifid_data.next_id.dtype
sifid_data.head()

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
1,SERENITY CT,8665,1550,8594,"(-85.6809503218, 38.1588670887)","(-85.6812346349, 38.1581804844)",NaN,NaN,NaN,NaN
2,S 28TH ST,5926,2854,2470,"(-85.8012012237, 38.230563793)","(-85.8013692698, 38.2293433595)",NaN,NaN,NaN,NaN
3,BEECH ST,473,6487,10596,"(-85.8050149882, 38.2289330216)","(-85.8048316249, 38.2275378873)",NaN,NaN,NaN,NaN
4,GARDEN DR,2442,13394,4702,"(-85.6802054534, 38.2480671366)","(-85.6798630078, 38.2476081599)",NaN,NaN,NaN,NaN
5,PARKWAY DR,4573,4162,8594,"(-85.7420397977, 38.2118147709)","(-85.7416475277, 38.2119685968)",NaN,NaN,NaN,NaN


In [267]:
# HERE
from collections import defaultdict

# algo to get links between segments in a roadway

data = sifid_data
test_sifid = 1

roadway = data[data.SIFID == test_sifid].copy() # copy for now
starts = defaultdict(list)

for segment_id, geofirst in roadway.GEOFIRST.items():
    starts[geofirst].append(segment_id)

for segment_id, geolast in roadway.GEOLAST.items():
    next_segment_ids = starts[geolast]

    if next_segment_ids: # ignore blank lists
        roadway.at[segment_id, 'next_id'] = next_segment_ids

    ends = defaultdict(list)
    # make prev_segment_ids
    for next_id in next_segment_ids:
        ends[next_id].append(segment_id)

    for next_id, prev_segment_ids in ends.items():
        roadway.at[next_id, 'prev_id'] = prev_segment_ids
    
roadway


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
2398,NO NAME,1,8594,8594,"(-85.8444815932, 38.2013753392)","(-85.844988314, 38.2000420639)",[30388],[30289],NaN,NaN
3506,NO NAME,1,7417,8594,"(-85.7096882714, 38.128131124)","(-85.7095108424, 38.1286888006)",[166804],NaN,NaN,NaN
6313,NO NAME,1,8594,591,"(-85.8953120366, 38.1440146893)","(-85.8920342109, 38.143922224)",[10250],[16431],NaN,NaN
8356,NO NAME,1,8594,8594,"(-85.8479877846, 38.1773413022)","(-85.8488128852, 38.1771472776)",[24435],[11064],NaN,NaN
8683,NO NAME,1,8594,8594,"(-85.8428031334, 38.2012149341)","(-85.8444815932, 38.2013753392)",[2398],[25924],NaN,NaN
10250,NO NAME,1,591,8594,"(-85.8920342109, 38.143922224)","(-85.8901779074, 38.1438753086)","[21005, 25324]",[6313],NaN,NaN
11063,NO NAME,1,8594,1813,"(-85.7086645165, 38.12493153)","(-85.7081067425, 38.1248562874)",NaN,NaN,NaN,NaN
11064,NO NAME,1,8594,8594,"(-85.8477406224, 38.1771966371)","(-85.8479877846, 38.1773413022)","[8356, 23118]",NaN,NaN,NaN
11492,NO NAME,1,8594,8594,"(-85.8485709622, 38.1776167113)","(-85.8492600142, 38.1776831385)",[28036],[23118],NaN,NaN
13954,NO NAME,1,8594,8594,"(-85.8428031334, 38.2012149341)","(-85.8434303311, 38.2008512248)","[28746, 30289]",[25924],NaN,NaN


In [268]:
# rework
# no default dicts; cleaner, probably faster
# # algo to get links between segments in a roadway

test_sifid = 1
roadway = data[data.SIFID == test_sifid].copy()
roadway_index = roadway.index

groupby_geofirst = roadway.groupby(by='GEOFIRST').groups

for segment_id, geolast in roadway.GEOLAST.items():
    next_segments = groupby_geofirst.get(geolast, None)
    if next_segments is not None: #ignore empties
        roadway.at[segment_id, 'next_id'] = tuple(next_segments)

groupby_geolast = roadway.groupby(by='GEOLAST').groups

for segment_id, geofirst in roadway.GEOFIRST.items():
    prev_segments = groupby_geolast.get(geofirst, None)
    if prev_segments is not None: # ignore empties
        roadway.at[segment_id, 'prev_id'] = tuple(prev_segments)


roadway




,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
2398,NO NAME,1,8594,8594,"(-85.8444815932, 38.2013753392)","(-85.844988314, 38.2000420639)","(30388,)","(8683, 30289)",NaN,NaN
3506,NO NAME,1,7417,8594,"(-85.7096882714, 38.128131124)","(-85.7095108424, 38.1286888006)","(166804,)",NaN,NaN,NaN
6313,NO NAME,1,8594,591,"(-85.8953120366, 38.1440146893)","(-85.8920342109, 38.143922224)","(10250,)","(16431,)",NaN,NaN
8356,NO NAME,1,8594,8594,"(-85.8479877846, 38.1773413022)","(-85.8488128852, 38.1771472776)","(24435,)","(11064,)",NaN,NaN
8683,NO NAME,1,8594,8594,"(-85.8428031334, 38.2012149341)","(-85.8444815932, 38.2013753392)","(2398,)","(25924,)",NaN,NaN
10250,NO NAME,1,591,8594,"(-85.8920342109, 38.143922224)","(-85.8901779074, 38.1438753086)","(21005, 25324)","(6313,)",NaN,NaN
11063,NO NAME,1,8594,1813,"(-85.7086645165, 38.12493153)","(-85.7081067425, 38.1248562874)",NaN,NaN,NaN,NaN
11064,NO NAME,1,8594,8594,"(-85.8477406224, 38.1771966371)","(-85.8479877846, 38.1773413022)","(8356, 23118)",NaN,NaN,NaN
11492,NO NAME,1,8594,8594,"(-85.8485709622, 38.1776167113)","(-85.8492600142, 38.1776831385)","(28036,)","(23118,)",NaN,NaN
13954,NO NAME,1,8594,8594,"(-85.8428031334, 38.2012149341)","(-85.8434303311, 38.2008512248)","(28746, 30289)","(25924,)",NaN,NaN


In [269]:
roadway
low_cross_sifs = sifid_data[sifid_data.SIFID.isin(roadway.SIFIDLOW) & (sifid_data.SIFIDHI == test_sifid)]
hi_cross_sifs = sifid_data[sifid_data.SIFID.isin(roadway.SIFIDHI) & (sifid_data.SIFIDLOW == test_sifid)]


cross_sifs = sifid_data[(sifid_data.SIFIDLOW == test_sifid) | (sifid_data.SIFIDHI == test_sifid)]
cross_sifs



,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
86,S FLOYD ST,2252,1,3191,"(-85.7483408726, 38.2530911319)","(-85.7483793493, 38.2529029274)",NaN,NaN,NaN,NaN
1332,BLUE WING DR,591,1,3925,"(-85.8920571895, 38.1432263537)","(-85.8920859804, 38.1424606241)",NaN,NaN,NaN,NaN
1893,EAGLE PASS,1813,203,1,"(-85.7079819882, 38.1254403376)","(-85.7081067425, 38.1248562874)",NaN,NaN,NaN,NaN
2389,BLUE WING DR,591,1,1,"(-85.8920342109, 38.143922224)","(-85.8920571895, 38.1432263537)",NaN,NaN,NaN,NaN
3653,ROCKFORD LN,5062,1644,1,"(-85.8482217831, 38.1784499755)","(-85.8486266793, 38.1786671473)",NaN,NaN,NaN,NaN
5361,CANE RUN RD,906,4397,1,"(-85.8124564765, 38.221092062)","(-85.8150314891, 38.2184316573)",NaN,NaN,NaN,NaN
5443,ROCKFORD LN,5062,1,5239,"(-85.8486266793, 38.1786671473)","(-85.8495054162, 38.1790719616)",NaN,NaN,NaN,NaN
8706,E JEFFERSON ST,3191,1,12883,"(-85.749252152, 38.2530094773)","(-85.7504029204, 38.253116398)",NaN,NaN,NaN,NaN
11220,CANE RUN RD,906,1,3925,"(-85.8967911478, 38.1433697959)","(-85.8969179309, 38.1426395313)",NaN,NaN,NaN,NaN
13587,CAMP GROUND RD,897,3628,1,"(-85.829307576, 38.2134911751)","(-85.8301351454, 38.2136097438)",NaN,NaN,NaN,NaN


In [270]:


low_groups = sifid_data.groupby(['SIFID', 'SIFIDLOW'])
hi_groups = sifid_data.groupby(['SIFID', 'SIFIDHI'])



In [271]:
display(ee.groups)

sifid_data.at[10250, 'GEOFIRST']

ee.get_group((1, 897))


{(-85.9368588569, 38.0059007504): [1334, 4613], (-85.9357035089, 38.0067117905): [11316], (-85.9332150252, 38.0090934509): [20599], (-85.9269230739, 38.0128719269): [2808, 21796], (-85.9214714039, 38.0185197301): [22217], (-85.9211733181, 38.0182948037): [15032, 25011], (-85.920226447, 38.0209466657): [22427], (-85.9177209754, 38.0217571424): [27330, 156207], (-85.9109850156, 38.0353251338): [760, 23539], (-85.9092247054, 38.020250973): [156206], (-85.9044306155, 38.0590019619): [30973], (-85.9039150286, 38.0732548479): [3872], (-85.9031568358, 38.0604441876): [14364, 22359], (-85.9024561559, 38.0755850425): [4688], (-85.9021426668, 38.0599151155): [27856], (-85.9021410913, 38.0161367636): [129328], (-85.9019691315, 38.0631753404): [31805], (-85.901791716, 38.0773288704): [26315], (-85.9017865047, 38.0619966831): [7484, 21733], (-85.9010371736, 38.0626921259): [1286, 3632], (-85.9009284252, 38.08037945): [6237, 16736], (-85.9005601728, 38.1647812505): [1507], (-85.9003728005, 38.056887

KeyError: (1, 897)

In [ ]:
# THERE
data = sifid_data
test_sifid = 1

roadway = data[data.SIFID == test_sifid].copy() # copy for now

lowsifids = set(roadway.SIFIDLOW)
ends = set(roadway.SIFIDHI)
starts = lowsifids - ends

ends -= starts
starts = list(starts)

print(starts)
for row in roadway[roadway.SIFIDLOW.isin(starts)].itertuples():
    next_row = roadway[roadway.GEOFIRST == row.GEOLAST]
    if len(next_row) != 1:
        continue
    else:
        next_row_id = next_row.index.item()

        roadway.at[row_id, 'next_id'] = next_row_id
        roadway.at[next_row_id, 'prev_id'] = row_id

        row_id = next_row_id
        geolast = next_row.GEOLAST.item()

        while 1:
            next_row = roadway[roadway.GEOFIRST == geolast]
            if len(next_row) != 1:
                break
            next_row_id = next_row.index.item()

            roadway.at[row_id, 'next_id'] = next_row_id
            roadway.at[next_row_id, 'prev_id'] = row_id

            row_id = next_row_id
            geolast = next_row.GEOLAST.item()
            



roadway
# this one is the problem
#      ROADNAME  SIFID  SIFIDLOW  SIFIDHI                         GEOFIRST  \
#8683   NO NAME      1      8594     8594  (-85.8428031334, 38.2012149341)   
#13954  NO NAME      1      8594     8594  (-85.8428031334, 38.2012149341)   



[897, 5062, 3623, 3405, 5487, 3191, 7417]


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
2398,NO NAME,1.0,8594.0,8594.0,"(-85.8444815932, 38.2013753392)","(-85.844988314, 38.2000420639)",NaN,NaN,NaN,NaN
3506,NO NAME,1.0,7417.0,8594.0,"(-85.7096882714, 38.128131124)","(-85.7095108424, 38.1286888006)",NaN,NaN,NaN,NaN
6313,NO NAME,1.0,8594.0,591.0,"(-85.8953120366, 38.1440146893)","(-85.8920342109, 38.143922224)",NaN,NaN,NaN,NaN
8356,NO NAME,1.0,8594.0,8594.0,"(-85.8479877846, 38.1773413022)","(-85.8488128852, 38.1771472776)",NaN,NaN,NaN,NaN
8683,NO NAME,1.0,8594.0,8594.0,"(-85.8428031334, 38.2012149341)","(-85.8444815932, 38.2013753392)",NaN,NaN,NaN,NaN
10250,NO NAME,1.0,591.0,8594.0,"(-85.8920342109, 38.143922224)","(-85.8901779074, 38.1438753086)",NaN,NaN,NaN,NaN
11063,NO NAME,1.0,8594.0,1813.0,"(-85.7086645165, 38.12493153)","(-85.7081067425, 38.1248562874)",NaN,NaN,NaN,NaN
11064,NO NAME,1.0,8594.0,8594.0,"(-85.8477406224, 38.1771966371)","(-85.8479877846, 38.1773413022)",NaN,NaN,NaN,NaN
11492,NO NAME,1.0,8594.0,8594.0,"(-85.8485709622, 38.1776167113)","(-85.8492600142, 38.1776831385)",NaN,NaN,NaN,NaN
13954,NO NAME,1.0,8594.0,8594.0,"(-85.8428031334, 38.2012149341)","(-85.8434303311, 38.2008512248)",NaN,NaN,NaN,NaN


In [ ]:

geo_first_gby = sifid_data.groupby(['SIFID', 'GEOFIRST'])
geo_last_gby = sifid_data.groupby(['SIFID', 'GEOLAST'])
geo_first = geo_first_gby.groups
geo_last = geo_last_gby.groups

In [ ]:

gf_group = sifid_data.groupby('GEOFIRST').groups
gl_group = sifid_data.groupby('GEOLAST').groups

pairs = list()

for gl_point, low_group in gl_group.items():
    hi_group = gf_group.get(gl_point, None)
    if hi_group is not None:
        ...
        #pairs.append((low_group.to_list(), hi
    
low_group.to_list()

[177346]

In [ ]:
class SIF_data:
    pass

class SIFID_data(SIF_data):
    def __init__(self, centerlines:pd.DataFrame):
        self.data = ... # select out data from centerlines raw
        self.connections = self.get_geo_connections()

    def get_geo_connections(self) -> pd.DataFrame:
        self.data


In [675]:
    
pd.DataFrame.from_records(pairs, columns=["low", "hi"])

def get_geo_connections(data):
    one_to_one = list()
    one_to_many = list()
    many_to_one = list()
    many_to_many = list()

    gby_geofirst = data.groupby('GEOFIRST').groups
    gby_geolast = data.groupby('GEOLAST').groups
    for geolast_point, low_group in gby_geolast.items():
        hi_group = gby_geofirst.get(geolast_point, None)
        if hi_group is not None: #skip empties
            low_group = low_group.to_list()
            hi_group = hi_group.to_list()
            if len(low_group) == 1:
                if len(hi_group) == 1:
                    one_to_one.append((low_group[0], hi_group[0]))
                else:
                    one_to_many.append((low_group[0], hi_group))
            elif len(hi_group) == 1:
                many_to_one.append((low_group, hi_group[0]))
            else: 
                many_to_many.append((low_group, hi_group))

    return {'one_to_one': one_to_one, 
            'one_to_many': one_to_many, 'many_to_one': many_to_one,
            'many_to_many': many_to_many}
    


connections = get_geo_connections(sifid_data)
connections

{'one_to_one': [(156207, 156206),
  (156206, 129328),
  (29275, 8031),
  (2204, 22633),
  (30369, 6422),
  (22149, 10149),
  (19931, 175758),
  (13233, 29442),
  (15408, 16510),
  (13484, 29261),
  (915, 29420),
  (27250, 12755),
  (163592, 163593),
  (113637, 132847),
  (132848, 166472),
  (17599, 30642),
  (29691, 12853),
  (41607, 166475),
  (25778, 22653),
  (11672, 20423),
  (9085, 20937),
  (20941, 32395),
  (8305, 14169),
  (178954, 178952),
  (25002, 23932),
  (6534, 22619),
  (22654, 31660),
  (30994, 30428),
  (7299, 14827),
  (156513, 156514),
  (20480, 561),
  (19707, 21915),
  (22972, 27911),
  (12643, 11644),
  (12895, 33050),
  (27674, 31852),
  (18344, 21169),
  (4333, 1020),
  (7841, 31033),
  (160388, 160389),
  (1020, 12715),
  (26866, 30529),
  (1710, 7036),
  (31621, 16534),
  (965, 14427),
  (14479, 12376),
  (140847, 140848),
  (130277, 130278),
  (8401, 32355),
  (26028, 32100),
  (155657, 442),
  (31890, 8806),
  (18965, 19788),
  (19961, 29365),
  (16045, 1310

In [702]:
cxda = pd.DataFrame(columns=['next_id', 'prev_id', 'hi_id', 'low_id', 'cx_hi', 'cx_low'])

def set_one_to_one_connection(data, low_index, hi_index):
    low_row = data.loc[low_index]
    hi_row = data.loc[hi_index]
    if low_row.SIFID == hi_row.SIFID:
        # same road way
        cxda.at[low_index, 'next_id'] = hi_index 
        cxda.at[hi_index, 'prev_id'] = low_index
    else:
        notfound = True
        if low_row.SIFIDHI == hi_row.SIFID:
            cxda.at[low_index, 'hi_id'] = hi_index
            cxda.at[hi_index, 'low_id'] = low_index
            notfound = False
        if hi_row.SIFIDHI == low_row.SIFID: # this can happen. *shrug*
            cxda.at[low_index, 'low_id'] = hi_index
            cxda.at[hi_index, 'hi_id'] = low_index
            notfound = False
        if notfound:
            cxda.at[low_index, 'cx_hi'] = hi_index
            cxda.at[hi_index, 'cx_low'] = low_index 
        

def set_one_to_many_connection(data, low_index, hi):
    ...

def set_many_to_one_connection(data, low, hi_index):
    ...

def set_many_to_many_connection(data, low, hi):
    ...


In [703]:
for lo, hi in connections['one_to_one']:
    set_one_to_one_connection(sifid_data, lo, hi)

cxda

,next_id,prev_id,hi_id,low_id,cx_hi,cx_low
156207,156206,NaN,NaN,NaN,NaN,NaN
156206,NaN,156207,129328,NaN,NaN,NaN
129328,NaN,NaN,NaN,156206,NaN,NaN
29275,NaN,NaN,8031,NaN,NaN,NaN
8031,NaN,NaN,NaN,29275,NaN,NaN
...,...,...,...,...,...,...
156193,156194,155653,NaN,NaN,NaN,NaN
156194,NaN,156193,NaN,NaN,NaN,NaN
38147,NaN,NaN,NaN,38158,NaN,NaN
108843,NaN,NaN,NaN,109820,NaN,NaN


In [699]:
display(cxda.cx_hi[cxda.cx_hi.notna()], 
        cxda.cx_low[cxda.cx_low.notna()])

27624    30777.0
Name: cx_hi, dtype: float64

30777    27624.0
Name: cx_low, dtype: float64

In [700]:
sifid_data.loc[[27624 ,   30777]]

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
27624,CANDACE WAY,8326,8325,8594,"(-85.7719600219, 38.1430445354)","(-85.771592421, 38.1424571006)",NaN,NaN,NaN,NaN
30777,CONNOR WAY,8327,8594,2672,"(-85.771592421, 38.1424571006)","(-85.7715273214, 38.1412880284)",NaN,NaN,NaN,NaN


In [694]:
cxda.loc[[30777,   27624,
17234,    23603,
160396   , 160397,
21726   ,   12028]]

,next_id,prev_id,hi_id,low_id,cx_hi,cx_low
30777,NaN,NaN,NaN,NaN,NaN,27624.0
27624,NaN,NaN,NaN,NaN,30777.0,NaN
17234,NaN,NaN,NaN,NaN,NaN,23603.0
23603,NaN,NaN,NaN,NaN,17234.0,NaN
160396,NaN,NaN,NaN,NaN,NaN,160397.0
160397,NaN,NaN,NaN,NaN,160396.0,NaN
21726,NaN,NaN,NaN,NaN,NaN,12028.0
12028,NaN,NaN,NaN,NaN,21726.0,NaN


In [ ]:


#display(lowdata, hidata)

fix_list = lambda xs: (xs if xs else None)

for index, value in lowdata.SIFID.items():
    comp = hidata[hidata.SIFID == value].index.tolist()
    #display(comp)
    comp2 = hidata[~hidata.index.isin(comp)].index.tolist() 

    display({'index':index, 'next_id':fix_list(comp), 'hi_id':fix_list(comp2)})
    
e = hidata.index.tolist()

da = dict()

#for ix, sifid in lowdata.SIFID.items():
#    display(ix, row)
#    seq_match = hidata[hidata.SIFID == sifid].index 
#    this = {'next_id': seq_match}
#    da[ix] = this
#da


low = lowdata.iloc[0]

hidata[hidata.SIFID == low.SIFID]

{'index': 1573, 'next_id': [10219], 'hi_id': [14380]}

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
14380,CASTLEVIEW DR,1001,6413,8594,"(-85.6391830222, 38.2612831118)","(-85.641012932, 38.2634883978)",NaN,NaN,NaN,NaN


In [654]:

cxda = pd.DataFrame(columns=['next_id','prev_id','low_id', 'hi_id', 'cx_low', 'cx_hi'])
setlist = lambda xs:(tuple(xs) if xs else None)



def fix(xs):
    length = len(xs)
    if length == 1:
        return xs[0]
    elif length:
        return list(xs)
    else:
        return pd.NA
        

def match_roadways(data, connection_pair, memo=cxda):
    low, hi = connection_pair
    lowlen = len(low) == 1 
    hilen = len(hi) == 1

    # one to one
    if lowlen and hilen:
        hi_index = hi[0]
        low_index = low[0]
        low_SIFID = data.at[low_index, 'SIFID']
        if data.at[hi_index, 'SIFID'] == low_SIFID:
            # same roadway
            cxda.at[low_index, 'next_id'] = hi_index
            cxda.at[hi_index, 'prev_id'] = low_index
            # probably shouldn't put the lists here directly might need to make a copy.
            # what happens if this list is modified somehow? Could that ever happen?
            # If now, it probably doesn't matter. 
            # This result data will get copied into another DataFrame at some point anyway.
        elif data.at[hi_index, 'SIFIDLOW'] == low_SIFID:
            # intersetion
            cxda.at[low_index, 'hi_id'] = hi_index
            cxda.at[hi_index, 'low_id'] = low_index
        else:
            # some other weird case: possibly no such entries
            cxda.at[low_index, 'cx_hi'] = hi_index
            cxda.at[hi_index, 'cx_low'] = low_index

    # one to many or many to one
    elif lowlen or hilen:
        # one low to many hi
        if lowlen:
            low_index = low[0]
            low_SIFID = data.at[low_index, 'SIFID']
            hi_rows = data.loc[hi]

            # get hi roadways, cross streets, other
            hi_roadways = hi_rows[hi_rows.SIFID == low_SIFID].index
            hi_cross = hi_rows[hi_rows.SIFIDLOW == low_SIFID].index
            hi_other = hi_rows.index.difference(hi_roadways.union(hi_cross))

            # set low data
            cxda.at[low_index, 'next_id'] = fix(hi_roadways)
            cxda.at[low_index, 'hi_id'] = fix(hi_cross)
            cxda.at[low_index, 'cx_hi'] = fix(hi_other)

            # set hi data
            for hi_index in hi_roadways:
                cxda.at[hi_index, 'prev_id'] = low_index
            for hi_index in hi_cross:
                cxda.at[hi_index, 'low_id'] = low_index
            for hi_index in hi_other:
                cxda.at[hi_index, 'cx_low'] = low_index
            #cxda.loc[hi_roadways, 'prev_id'] = low_index
            #cxda.loc[hi_cross, 'low_id'] = low_index
            #cxda.loc[hi_other, 'cx_low'] = low_index

        # many low to one hi
        else:
            hi_index = hi[0]
            hi_SIFID = data.at[hi_index, 'SIFID']
            low_rows = data.loc[low]

            # get low roadways, cross streets, other
            low_roadways = low_rows[low_rows.SIFID == hi_SIFID].index
            low_cross = low_rows[low_rows.SIFIDHI == hi_SIFID].index
            low_other = low_rows.index.difference(low_roadways.union(low_cross))

            # set hi data
            cxda.at[hi_index, 'prev_id'] = fix(low_roadways)
            cxda.at[hi_index, 'low_id'] = fix(low_cross)
            cxda.at[hi_index, 'cx_low'] = fix(low_other)

            # set low data
            for low_index in low_roadways:
                cxda.at[hi_index, 'next_id'] = hi_index
            for low_index in low_cross:
                cxda.at[hi_index, 'hi_id'] = hi_index
            for low_index in low_other:
                cxda.at[hi_index, 'cx_hi'] = hi_index

    # many low to many hi 
    else:
        low_rows = data.loc[low].SIFID
        hi_rows = data.loc[hi][['SIFID', 'SIFIDLOW']]

        for low_index, low_SIFID in low_rows.items():
            low_roadways = hi_rows[hi_rows.SIFID == low_SIFID].index
            low_cross = hi_rows[hi_rows.SIFIDLOW == low_SIFID].index
            low_other = low_roadways.union(low_cross).symmetric_difference(hi)

            cxda.at[low_index, 'next_id'] = fix(low_roadways)
            cxda.at[low_index, 'hi_id'] = fix(low_cross)
            cxda.at[low_index, 'cx_hi'] = fix(low_other)

        for hi_index, hi_SIFID, hi_SIFIDLOW in hi_rows.itertuples():
            hi_roadways = low_rows[low_rows == hi_SIFID].index
            hi_cross = low_rows[low_rows == hi_SIFIDLOW].index
            hi_other = hi_roadways.union(hi_cross).symmetric_difference(low)

            cxda.at[hi_index, 'prev_id'] = fix(hi_roadways)
            cxda.at[hi_index, 'low_id'] = fix(hi_cross)
            cxda.at[hi_index, 'cx_low'] = fix(hi_other)


            

        
        
                
                
            

        

In [655]:

"""
if lowlen:
    low_index = low[0]
    low_SIFID = data.at[low_index, "SIFID"]
    ...
    rwy =hi[hi.SIFID == low_SIFID]
    rwy[prev] == this
    cross =hi[hi.SIFIDLOW == low_SIFID]
    other = hi[~(rwy + cross)]
""" 

for low, hi in connections:
    match_roadways(sifid_data, (low, hi))

cxda

,next_id,prev_id,low_id,hi_id,cx_low,cx_hi
11316,11316,21796,20599,11316,<NA>,<NA>
4613,NaN,11316,NaN,NaN,NaN,NaN
1334,NaN,NaN,11316,NaN,NaN,NaN
15032,21796,27330,NaN,2808,NaN,<NA>
21796,NaN,15032,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
177349,108844,177348,NaN,177347,NaN,<NA>
108844,NaN,177349,NaN,NaN,NaN,NaN
177347,NaN,NaN,177349,NaN,NaN,NaN
177348,177349,NaN,NaN,177040,NaN,<NA>


In [659]:
cxda.cx_low.value_counts()

cx_low
19532     2
29529     2
164237    2
27573     2
20410     2
         ..
15064     1
13157     1
33114     1
5500      1
1932      1
Name: count, Length: 881, dtype: int64

In [657]:
cxda.cx_low[~cxda.cx_low.isna()]

27856      27017
26259      10162
2266        5164
25824      11250
22739     146278
           ...  
38130      37473
1541        1542
1567      104687
123873     10584
44807       1932
Name: cx_low, Length: 1101, dtype: object

In [665]:
test = [27856 , 27017]
test = [44807     ,  1932]

display(
sifid_data.loc[test, :],
cxda.loc[test, :])
#sifid_data.loc[[44807   ,    1932],:]

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
44807,ASHWORTH LN,12632,12804,10745,"(-85.4139452562, 38.2369794317)","(-85.4163812062, 38.2373828508)",NaN,NaN,NaN,NaN
1932,NOTTING HILL BLVD,12583,12595,12804,"(-85.4124007712, 38.2367036922)","(-85.4139452562, 38.2369794317)",NaN,NaN,NaN,NaN


,next_id,prev_id,low_id,hi_id,cx_low,cx_hi
44807,NaN,<NA>,1727,44807,1932,44807
1932,NaN,20271,NaN,NaN,NaN,NaN


In [519]:
xs = sifid_data.loc[102:105].copy()

ys = sifid_data.loc[105:107].copy()

display(xs, ys)

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
102,W CHESTNUT ST,1132,5608,1870,"(-85.7659874323, 38.2498497625)","(-85.764499876, 38.2496818796)",NaN,NaN,NaN,NaN
103,HASKIN AVE,2767,3708,3525,"(-85.7867074545, 38.1729543037)","(-85.7889057879, 38.1732430068)",NaN,NaN,NaN,NaN
104,PEACHTREE AVE,4597,889,474,"(-85.7795133347, 38.1925305777)","(-85.7795077699, 38.1916873341)",NaN,NaN,NaN,NaN
105,WOODRIDGE LAKE BLVD,10332,12657,10756,"(-85.860284218, 38.0911579781)","(-85.8613810886, 38.091199465)",NaN,NaN,NaN,NaN


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOFIRST,GEOLAST,next_id,prev_id,low_id,hi_id
105,WOODRIDGE LAKE BLVD,10332,12657,10756,"(-85.860284218, 38.0911579781)","(-85.8613810886, 38.091199465)",NaN,NaN,NaN,NaN
106,WOODRIDGE LAKE BLVD,10332,10329,12657,"(-85.8594025126, 38.0924017026)","(-85.860284218, 38.0911579781)",NaN,NaN,NaN,NaN
107,SAND PIT LN,12657,10332,8594,"(-85.860284218, 38.0911579781)","(-85.8592079035, 38.0897552569)",NaN,NaN,NaN,NaN


In [527]:
xs.index.difference(ys.index)

Index([102, 103, 104], dtype='int64')